# IEX vs Okta Checker
So sánh trạng thái agent trong Okta snapshot với lịch IEX (dữ liệu đã qua cleaner).

In [152]:
import pandas as pd
from datetime import datetime

# Hiển thị đủ số dòng và số cột mong muốn
pd.set_option("display.max_rows", 100)   # tối đa số dòng hiển thị
pd.set_option("display.max_columns", None)  # hiện tất cả các cột
pd.set_option("display.width", None)   # không giới hạn độ rộng

In [153]:
# Đọc dữ liệu từ file đã clean (iex_cleaner xuất ra)
iex_df = pd.read_excel('iex-data-extracted.xlsx')
okta_df = pd.read_csv('okta.csv')

# Xóa cột "Available On" nếu tồn tại
if "Available On" in okta_df.columns:
    okta_df = okta_df.drop(columns=["Available On"])

iex_df.head(59)

,IEX Id,Name,Shift,Date,Activity,Start time,End time,Site,Supervisor Name,LOB,Email Id
0,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-27,Open Time,2025-08-27 13:00:00,2025-08-27 14:10:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
1,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-27,Break,2025-08-27 14:10:00,2025-08-27 14:25:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
2,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-27,Open Time,2025-08-27 14:25:00,2025-08-27 16:00:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
3,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-27,Lunch,2025-08-27 16:00:00,2025-08-27 17:00:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
4,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-27,Open Time,2025-08-27 17:00:00,2025-08-27 20:40:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
5,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-27,Break,2025-08-27 20:40:00,2025-08-27 20:55:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
6,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-27,Open Time,2025-08-27 20:55:00,2025-08-27 22:00:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
7,3085733,"BUI, MINHAN",6:00 AM - 3:00 PM,2025-08-27,Open Time,2025-08-27 06:00:00,2025-08-27 08:45:00,ONEHUB,To Anh Duy,Lodging,minhan.bui@concentrix.com
8,3085733,"BUI, MINHAN",6:00 AM - 3:00 PM,2025-08-27,Break,2025-08-27 08:45:00,2025-08-27 09:00:00,ONEHUB,To Anh Duy,Lodging,minhan.bui@concentrix.com
9,3085733,"BUI, MINHAN",6:00 AM - 3:00 PM,2025-08-27,Open Time,2025-08-27 09:00:00,2025-08-27 10:10:00,ONEHUB,To Anh Duy,Lodging,minhan.bui@concentrix.com


In [154]:
# Đổi tên cột Okta cho đồng bộ (nếu cần chỉnh lại tuỳ dataset)
okta_df = okta_df.rename(columns={
    'userName': 'Name',
    'status': 'Activity',
    'duration': 'Duration'
})
okta_df['CheckTime'] = datetime.now()
okta_df.head()

,Agent Name,Duration,State,Assigned Workitem Count,Agent Email,Queue Group / Routing Profile,Forecast Group,Manager Email,Business Location,CheckTime
0,"Bui, Ngoc Thuan Vy",00:08:44,TRAINING,NaN,ngocthuanvy.bui@concentrix.com,Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,thienkim.chau@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-28 01:16:30.660192
1,"Bui, The Anh",00:59:49,LUNCH,NaN,theanh.bui1@concentrix.com,Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,thienkim.chau@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-28 01:16:30.660192
2,"Bui, Thi Lan Anh",08:22:39,ENDOFSHIFT,NaN,thilananh.bui1@concentrix.com,Chat_OD_EN_Car_Activity,GEN_GEN_EN_GCS_GLG_CHT,huynhductran.tran@concentrix,Concentrix (Ho Chi Minh City),2025-08-28 01:16:30.660192
3,"Bui, Thi Ngoc Tram",00:02:50,AVAILABLECHAT,NaN,thingoctram.bui@concentrix.com,Chat_OD_EN_Car_Activity,GEN_GEN_EN_GCS_GLG_CHT,chihuy.vong@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-28 01:16:30.660192
4,"Chau, Thien Kim",00:50:17,ENDOFSHIFT,NaN,thienkim.chau@concentrix.com,Chat_OD_EN_Dual_GDS,GEN_GEN_EN_GCS_GNL_CHT,kirpan.patar@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-28 01:16:30.660192


In [155]:
# Chuyển tất cả giá trị "Open Time" trong cột Activity thành "AVAILABLECHAT"
iex_df["Activity"] = iex_df["Activity"].replace("Open Time", "AVAILABLECHAT")

# Chuyển Start/End về datetime
iex_df['Start time'] = pd.to_datetime(iex_df['Start time'], errors='coerce')
iex_df['End time']   = pd.to_datetime(iex_df['End time'], errors='coerce')
iex_df.head()

,IEX Id,Name,Shift,Date,Activity,Start time,End time,Site,Supervisor Name,LOB,Email Id
0,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-27,AVAILABLECHAT,2025-08-27 13:00:00,2025-08-27 14:10:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
1,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-27,Break,2025-08-27 14:10:00,2025-08-27 14:25:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
2,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-27,AVAILABLECHAT,2025-08-27 14:25:00,2025-08-27 16:00:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
3,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-27,Lunch,2025-08-27 16:00:00,2025-08-27 17:00:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
4,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-27,AVAILABLECHAT,2025-08-27 17:00:00,2025-08-27 20:40:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com


In [156]:
import re

def clean_name(name: str) -> str:
    if pd.isna(name):
        return name
    # Thay dấu phẩy bằng khoảng trắng
    name = name.replace(",", " ")
    # Thêm khoảng trắng trước chữ in hoa (trừ chữ cái đầu)
    name = re.sub(r'(?<!^)(?=[A-Z])', ' ', name)
    # Chuẩn hoá khoảng trắng thừa
    name = " ".join(name.split())
    return name.strip()

# Chuẩn hoá cho cả IEX và Okta
iex_df["Name"] = iex_df["Name"].astype(str).map(clean_name)
okta_df["Agent Name"] = okta_df["Agent Name"].astype(str).map(clean_name)
#iex_df.head()
okta_df.head()

,Agent Name,Duration,State,Assigned Workitem Count,Agent Email,Queue Group / Routing Profile,Forecast Group,Manager Email,Business Location,CheckTime
0,Bui Ngoc Thuan Vy,00:08:44,TRAINING,NaN,ngocthuanvy.bui@concentrix.com,Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,thienkim.chau@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-28 01:16:30.660192
1,Bui The Anh,00:59:49,LUNCH,NaN,theanh.bui1@concentrix.com,Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,thienkim.chau@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-28 01:16:30.660192
2,Bui Thi Lan Anh,08:22:39,ENDOFSHIFT,NaN,thilananh.bui1@concentrix.com,Chat_OD_EN_Car_Activity,GEN_GEN_EN_GCS_GLG_CHT,huynhductran.tran@concentrix,Concentrix (Ho Chi Minh City),2025-08-28 01:16:30.660192
3,Bui Thi Ngoc Tram,00:02:50,AVAILABLECHAT,NaN,thingoctram.bui@concentrix.com,Chat_OD_EN_Car_Activity,GEN_GEN_EN_GCS_GLG_CHT,chihuy.vong@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-28 01:16:30.660192
4,Chau Thien Kim,00:50:17,ENDOFSHIFT,NaN,thienkim.chau@concentrix.com,Chat_OD_EN_Dual_GDS,GEN_GEN_EN_GCS_GNL_CHT,kirpan.patar@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-28 01:16:30.660192


In [157]:
# Đọc file dimension-headcount
hc_df = pd.read_excel(
    r"C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Vu\unscheduled-agent\dimension-headcount.xlsx"
)

# Chuẩn hóa IEX ID trong headcount
hc_df["IEX ID"] = (
    hc_df["IEX ID"]
    .astype(str)
    .str.strip()
    .str.split(".").str[0]   # bỏ .0
)

# Merge okta_df với headcount để lấy thêm IEX ID
okta_df = okta_df.merge(
    hc_df[["IEX ID", "Email Id"]],
    left_on="Agent Email",
    right_on="Email Id",
    how="left"
).drop(columns=["Email Id"])

# Đảm bảo IEX ID trong okta_df cũng ở dạng string số nguyên
okta_df["IEX ID"] = (
    okta_df["IEX ID"]
    .astype(str)
    .str.strip()
    .str.split(".").str[0]
)

okta_df.head()

,Agent Name,Duration,State,Assigned Workitem Count,Agent Email,Queue Group / Routing Profile,Forecast Group,Manager Email,Business Location,CheckTime,IEX ID
0,Bui Ngoc Thuan Vy,00:08:44,TRAINING,NaN,ngocthuanvy.bui@concentrix.com,Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,thienkim.chau@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-28 01:16:30.660192,3109394
1,Bui The Anh,00:59:49,LUNCH,NaN,theanh.bui1@concentrix.com,Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,thienkim.chau@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-28 01:16:30.660192,3109498
2,Bui Thi Lan Anh,08:22:39,ENDOFSHIFT,NaN,thilananh.bui1@concentrix.com,Chat_OD_EN_Car_Activity,GEN_GEN_EN_GCS_GLG_CHT,huynhductran.tran@concentrix,Concentrix (Ho Chi Minh City),2025-08-28 01:16:30.660192,3084735
3,Bui Thi Ngoc Tram,00:02:50,AVAILABLECHAT,NaN,thingoctram.bui@concentrix.com,Chat_OD_EN_Car_Activity,GEN_GEN_EN_GCS_GLG_CHT,chihuy.vong@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-28 01:16:30.660192,3093456
4,Chau Thien Kim,00:50:17,ENDOFSHIFT,NaN,thienkim.chau@concentrix.com,Chat_OD_EN_Dual_GDS,GEN_GEN_EN_GCS_GNL_CHT,kirpan.patar@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-28 01:16:30.660192,nan


In [158]:
# Chuẩn hóa IEX ID trong okta_df
okta_df["IEX ID"] = (
    okta_df["IEX ID"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)  # bỏ .0 nếu có
    .str.strip()
)
okta_df = okta_df[okta_df["IEX ID"].str.lower() != "nan"]  # loại bỏ nan string

# Chuẩn hóa IEX Id trong iex_df
iex_df["IEX Id"] = (
    iex_df["IEX Id"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)

In [159]:
print("okta IEX ID sample:", okta_df["IEX ID"].unique()[:10])
print("iex IEX Id sample:", iex_df["IEX Id"].unique()[:10])

okta IEX ID sample: ['3109394' '3109498' '3084735' '3093456' '3096524' '3096692' '3112765'
 '3091393' '3082332' '3089163']
iex IEX Id sample: ['3052306' '3085733' '3092836' '3097227' '3109394' '3109498' '3093456'
 '3089245' '3091844' '3096524']


In [160]:
# 1. So sánh giữa Okta và IEX bằng IEX ID
results = []
now = datetime.now()

for _, row in okta_df.iterrows():
    iex_id = row["IEX ID"]  # đã chuẩn hóa sang string
    activity_okta = row["State"]
    workitem_count = row.get("Assigned Workitem Count", None)
    agent_email = row.get("Agent Email", None)

    # Skip các trường hợp không phải Available chat nhưng productive
    if str(activity_okta).strip().upper() != "AVAILABLECHAT" and pd.notna(workitem_count):
        continue

    # Tìm agent trong IEX dựa trên IEX ID
    iex_agent = iex_df[iex_df["IEX Id"] == iex_id]
    iex_now = iex_agent[(iex_agent["Start time"] <= now) & (iex_agent["End time"] >= now)]

    if not iex_now.empty:
        row_iex = iex_now.iloc[0]
        activity_iex = row_iex["Activity"]
        start_iex = row_iex["Start time"]
        end_iex = row_iex["End time"]
        site = row_iex.get("Site", None)
        lob = row_iex.get("LOB", None)
        supervisor = row_iex.get("Supervisor Name", None)
    else:
        activity_iex = "N/A"
        start_iex = None
        end_iex = None
        site = None
        lob = None
        supervisor = None

    results.append({
        "IEX ID": iex_id,
        "Agent": row["Agent Name"],   # giữ nguyên tên từ Okta
        "LOB": lob,
        "Site": site,
        "Agent Email": agent_email,
        "Supervisor Name": supervisor,
        "Activity_Okta": activity_okta,
        "Activity_IEX": activity_iex,
        "Start_IEX": start_iex,
        "End_IEX": end_iex,
        "Match": str(activity_okta).strip().lower() == str(activity_iex).strip().lower()
    })


# 2. Agent có trong IEX nhưng không có trong Okta
iex_current = iex_df[
    (iex_df["Start time"] <= now) & 
    (iex_df["End time"] >= now)
].copy()

iex_current = iex_current[
    iex_current["Activity"].notna() & (iex_current["Activity"].astype(str).str.strip() != "")
]

iex_current = iex_current.sort_values(["IEX Id", "Start time"]).drop_duplicates(subset=["IEX Id"], keep="first")

okta_ids = set(okta_df["IEX ID"])

for _, row in iex_current.iterrows():
    if row["IEX Id"] not in okta_ids:
        results.append({
            "IEX ID": row["IEX Id"],
            "Agent": row["Name"],  
            "LOB": row.get("LOB", None),
            "Site": row.get("Site", None),
            "Agent Email": None,  # vì không có trong Okta nên để None
            "Supervisor Name": row.get("Supervisor Name", None),
            "Activity_Okta": "N/A",
            "Activity_IEX": row["Activity"],
            "Start_IEX": row["Start time"],
            "End_IEX": row["End time"],
            "Match": False
        })


# 3. Kết quả cuối
result_df = pd.DataFrame(results)

exclude_activities = ["Termination", "No Call/No Show", "Unpaid Leave", "PTO"]
result_df = result_df[~result_df["Activity_IEX"].isin(exclude_activities)]

# Thứ tự cột mới: IEX ID trước Agent
col_order = ["IEX ID", "Agent", "LOB", "Site", "Agent Email", "Supervisor Name",
             "Activity_Okta", "Activity_IEX", "Start_IEX", "End_IEX", "Match"]

result_df = result_df[col_order]

# Sort theo LOB rồi Site
result_df = result_df.sort_values(by=["LOB", "Site"], ascending=[True, True]).reset_index(drop=True)


# Ép IEX ID về dạng số nguyên (loại bỏ .0), sau đó về string
result_df["IEX ID"] = (
    result_df["IEX ID"]
    .astype(str)        # chuyển về string
    .str.split(".").str[0]  # bỏ phần .0 nếu có
    .str.strip()
)

result_df

,IEX ID,Agent,LOB,Site,Agent Email,Supervisor Name,Activity_Okta,Activity_IEX,Start_IEX,End_IEX,Match
0,3093456,Bui Thi Ngoc Tram,Lodging,FLEMINGTON,thingoctram.bui@concentrix.com,Chí Huy,AVAILABLECHAT,AVAILABLECHAT,2025-08-28 01:15:00,2025-08-28 03:50:00,True
1,3096692,Dang Anh Trung,Lodging,FLEMINGTON,anhtrung.dang@concentrix.com,Chí Huy,AVAILABLECHAT,Lunch,2025-08-28 01:00:00,2025-08-28 02:00:00,False
2,3091393,Dang Phuong Tien,Lodging,FLEMINGTON,phuongtien.dang1@concentrix.com,Jimi Kurt (Nguyễn Khôi),LUNCH,AVAILABLECHAT,2025-08-28 00:00:00,2025-08-28 02:20:00,False
3,3092444,Le Thanh Tung,Lodging,FLEMINGTON,thanhtung.le@concentrix.com,Jimi Kurt (Nguyễn Khôi),AVAILABLECHAT,AVAILABLECHAT,2025-08-28 00:30:00,2025-08-28 01:30:00,True
4,3091855,Nguyen Ngan Giang,Lodging,FLEMINGTON,ngangiang.nguyen1@concentrix.com,Do Hong Hanh,AVAILABLECHAT,AVAILABLECHAT,2025-08-28 00:00:00,2025-08-28 02:20:00,True
5,3090603,Nguyen Thi Thanh Tuyen,Lodging,FLEMINGTON,thithanhtuyen.nguyen3@concentrix.com,Jimi Kurt (Nguyễn Khôi),AVAILABLECHAT,Lunch,2025-08-28 01:05:00,2025-08-28 02:05:00,False
6,3091294,Nguyen Xuan Quynh,Lodging,FLEMINGTON,xuanquynh.nguyen1@concentrix.com,Jimi Kurt (Nguyễn Khôi),AVAILABLECHAT,AVAILABLECHAT,2025-08-28 00:15:00,2025-08-28 01:25:00,True
7,3093608,Phan Tran Chanh,Lodging,FLEMINGTON,tranchanh.phan@concentrix.com,Chí Huy,TRAINING,AVAILABLECHAT,2025-08-28 00:30:00,2025-08-28 02:55:00,False
8,3096746,Ton Nu Huynh Nhi,Lodging,FLEMINGTON,nuhuynhnhi.ton@concentrix.com,Chí Huy,AVAILABLECHAT,Lunch,2025-08-28 01:00:00,2025-08-28 02:00:00,False
9,3091353,Trieu Gia V Inh,Lodging,FLEMINGTON,giavinh.trieu@concentrix.com,Jimi Kurt (Nguyễn Khôi),AVAILABLECHAT,Lunch,2025-08-28 01:15:00,2025-08-28 02:15:00,False


In [161]:
# Xuất mismatch ra file Excel
mismatch_df = result_df[(result_df['Match'] == False) & (result_df["Activity_IEX"] != "N/A")]
mismatch_df.to_excel('iex_okta_mismatch.xlsx', index=False)


#mismatch_df = mismatch_df[mismatch_df["LOB"] == "Lodging"]
#mismatch_df = mismatch_df[mismatch_df["Site"] == "FLEMINGTON"]

columns = ["Agent", "Activity_Okta", "Activity_IEX", "Start_IEX", "End_IEX", "Match"]

mismatch_df[columns]

,Agent,Activity_Okta,Activity_IEX,Start_IEX,End_IEX,Match
1,Dang Anh Trung,AVAILABLECHAT,Lunch,2025-08-28 01:00:00,2025-08-28 02:00:00,False
2,Dang Phuong Tien,LUNCH,AVAILABLECHAT,2025-08-28 00:00:00,2025-08-28 02:20:00,False
5,Nguyen Thi Thanh Tuyen,AVAILABLECHAT,Lunch,2025-08-28 01:05:00,2025-08-28 02:05:00,False
7,Phan Tran Chanh,TRAINING,AVAILABLECHAT,2025-08-28 00:30:00,2025-08-28 02:55:00,False
8,Ton Nu Huynh Nhi,AVAILABLECHAT,Lunch,2025-08-28 01:00:00,2025-08-28 02:00:00,False
9,Trieu Gia V Inh,AVAILABLECHAT,Lunch,2025-08-28 01:15:00,2025-08-28 02:15:00,False
10,Bui Ngoc Thuan Vy,TRAINING,AVAILABLECHAT,2025-08-28 00:00:00,2025-08-28 02:00:00,False
12,Chinh Ngoc Thu,TRAINING,AVAILABLECHAT,2025-08-28 00:00:00,2025-08-28 02:40:00,False
14,Dinh Thi Ngoc Han,AVAILABLECHAT,Lunch,2025-08-28 01:10:00,2025-08-28 02:10:00,False
16,Duong Thi Thuy Duong,LUNCH,AVAILABLECHAT,2025-08-28 01:10:00,2025-08-28 03:55:00,False


In [162]:
# --- Export full comparison with both True/False ---
try:
    out_file = 'iex_okta_comparison.xlsx'
    result_df.to_excel(out_file, index=False)
    print(f'Saved full comparison to: {out_file}')
    result_df
except NameError as e:
    print('result_df is not defined. Please run the comparison cells above first.')
    raise


Saved full comparison to: iex_okta_comparison.xlsx
